# E002 — Audio: MFCC + delta + delta-delta

Identical to E001 except `extract_mfcc` now appends Δ and ΔΔ features (13 → 39 per frame).
Everything else — GMM sizes, fold spec, seed — is unchanged so the result is directly comparable.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

In [ ]:
# The only change vs E001: append delta and delta-delta
def extract_mfcc(wav_path: Path, n_mfcc: int = 13) -> np.ndarray:
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)   # (n_mfcc, T)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T                 # (T, 39)
    mfcc  -= mfcc.mean(axis=0)                                  # CMN
    return mfcc


def extract_mfcc_batch(df, folder: Path):
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
            wav = folder / sf / (row["stem"] + ".wav")
            if wav.exists():
                break
        mfcc = extract_mfcc(wav)
        all_mfcc.append(mfcc)
        all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


def train_gmm(X, n_components, seed=67):
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def score_utterance(wav_path, gmm_target, gmm_nontarget):
    mfcc = extract_mfcc(wav_path)
    return float((gmm_target.score_samples(mfcc) - gmm_nontarget.score_samples(mfcc)).mean())


# Sanity check
sample = manifest.iloc[0]
for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
    w = DATA / sf / (sample["stem"] + ".wav")
    if w.exists(): break
print(f"Feature shape: {extract_mfcc(w).shape}  (expected T × 39)")

In [ ]:
N_MFCC = 13
N_TARGET_COMPONENTS = 8
N_NONTARGET_COMPONENTS = 32
SEED = 67

oof_scores = np.full(len(manifest), np.nan)
fold_results = []

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    print(f"Fold {fold_id}: train={len(train_df)}, val={len(val_df)}")

    X_train, y_train = extract_mfcc_batch(train_df, DATA)
    gmm_t  = train_gmm(X_train[y_train == 1], N_TARGET_COMPONENTS,    SEED)
    gmm_nt = train_gmm(X_train[y_train == 0], N_NONTARGET_COMPONENTS, SEED)

    for idx, row in val_df.iterrows():
        for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
            wav = DATA / sf / (row["stem"] + ".wav")
            if wav.exists(): break
        oof_scores[idx] = score_utterance(wav, gmm_t, gmm_nt)

    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    print(f"  → EER = {eer*100:.2f}%, min-DCF = {min_dcf:.4f}")

print("\nDone.")

In [ ]:
import pandas as pd

results_df  = pd.DataFrame(fold_results)
mean_eer    = results_df["eer"].mean()
std_eer     = results_df["eer"].std()
mean_dcf    = results_df["min_dcf"].mean()
std_dcf     = results_df["min_dcf"].std()

print("=" * 45)
print(f"{'Fold':<8} {'EER [%]':>10} {'min-DCF':>10}")
print("-" * 45)
for _, r in results_df.iterrows():
    print(f"{int(r.fold):<8} {r.eer*100:>10.2f} {r.min_dcf:>10.4f}")
print("-" * 45)
print(f"{'mean±std':<8} {mean_eer*100:>7.2f}±{std_eer*100:.2f} {mean_dcf:>7.4f}±{std_dcf:.4f}")
print("=" * 45)

y_all = manifest["label"].to_numpy()
eer_oof, _   = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"\nOOF overall: EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}, threshold = {thr:.3f}")

print(f"\nE001 baseline: EER = 17.92 ± 7.81%, min-DCF = 0.2250")
print(f"E002 deltas:   EER = {mean_eer*100:.2f} ± {std_eer*100:.2f}%, min-DCF = {mean_dcf:.4f}")
delta_eer = mean_eer*100 - 17.92
print(f"Change: {delta_eer:+.2f}% EER  ({'improvement' if delta_eer < 0 else 'regression'})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bins = np.linspace(oof_scores.min(), oof_scores.max(), 40)
ax.hist(oof_scores[y_all==0], bins=bins, alpha=0.6, color="steelblue", label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bins, alpha=0.7, color="tomato",    label="target",     density=True)
ax.axvline(thr, color="green", ls="--", lw=2, label=f"threshold = {thr:.2f}")
ax.set_xlabel("OOF score (LLR)")
ax.set_title("Score distributions")
ax.legend()

ax = axes[1]
fpr, tpr, _ = roc_curve(y_all, oof_scores)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color="tomato", lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlabel("FAR")
ax.set_ylabel("1 - FRR")
ax.set_title("ROC curve (OOF)")
ax.legend()

plt.suptitle(f"E002 — MFCC+Δ+ΔΔ  |  EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}")
plt.tight_layout()
plt.show()